LangChain 

### Агент: генератор тестовых данных для SQL-задач

Я недавно делал задачи по курсу продуктовой аналитики, там были много задач на sql, где было описание схем и того, что нужно сделать в задаче, но не было данных, было бы здорово сгенерировать данные под задачу с учетом крайних случаев, чтобы проверить свое решение.

### Как работает:

Агент построен по принципу Reason + Act

Reason - модель анализирует задачу: какие данные нужны, какие крайние случаи важны, сколько строк достаточно

Act - Вызывает следующие инструменты:
- validate-sql - проверяет сгенерированный sql
- save_sql_script - сохраняет скрипт в sql файл
- final_answer - завершает работу и возвращает результат

Memory - агент помнит историю диалога, если нужно что-то подправить





In [ ]:
OPENROUTER_TOKEN = ""

In [11]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    api_key=OPENROUTER_TOKEN,
    base_url="https://openrouter.ai/api/v1",
    model="openrouter/free"
)

print(llm.invoke("Привет! Ответь одним словом.").content)

Привет!


Tools

In [ ]:
import sqlite3
from langchain_core.tools import tool

@tool
def validate_sql(sql_script: str) -> str:
    """
    Проверяет синтаксис SQL-скрипта через SQLite.
    Используй перед save_sql_script чтобы убедиться что скрипт корректен.
    Возвращает 'SQL корректен' или описание ошибки.
    """
    
    context = sqlite3.connect(":memory:")
    try:
        context.executescript(sql_script)
        return f"SQL корректен"
    except Exception as e:
        return f"Ошибка: {e}"
    finally:
        context.close()

@tool
def save_sql_script(filename: str, sql_content: str) -> str:
    """
    Сохраняет SQL-скрипт в файл.
    Используй только после успешной валидации через validate_sql.

    filename: имя файла без пути, например 'task1.sql'
    sql_content: полный SQL-скрипт (CREATE TABLE + INSERT)
    """
    
    with open(filename, "w", encoding="utf-8") as f:
        f.write(sql_content)
    return f"Скрипт сохранён в '{filename}'"

@tool
def final_answer(answer: str, tools_used: list[str]):
    """
    Используй этот инструмент чтобы вернуть финальный ответ пользователю.
    answer: текст ответа на естественном языке
    tools_used: список инструментов которые были использованы
    """
    return {"answer": answer, "tools_used": tools_used}

tools = [validate_sql, save_sql_script, final_answer]

Промпт

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.base import RunnableSerializable

CASE_HINTS = """
group_by: группа с одним элементом, NULL в группируемом поле,
одинаковые значения агрегатной функции у разных групп

join: запись есть только в одной из таблиц, несколько совпадений,
NULL в поле по которому делается JOIN

subquery:  подзапрос возвращает пустой результат, дублирующиеся значения

window: одинаковые значения для ранжирования, одна строка в секции,
NULL в ORDER BY внутри окна

filter: граничное значение, NULL в фильтруемом поле,
пустой результат после фильтрации
"""

agent_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Ты — помощник для подготовки к SQL-собеседованиям. "
     "Твоя задача: по описанию задачи и схеме таблиц сгенерировать скрипт "
     "с тестовыми данными — только CREATE TABLE и INSERT-ы.\n"
     "Алгоритм работы:\n"
     "1. Сгенерируй скрипт: CREATE TABLE, 10-15 записей в INSERT-ах\n"
     "2. Вызови validate_sql — убедись что скрипт синтаксически корректен\n"
     "3. Если validate_sql вернул ошибку — исправь и проверь снова\n"
     "4. Вызови save_sql_script чтобы сохранить скрипт в файл\n"
     "5. Вызови final_answer чтобы сообщить пользователю результат\n"
     "Важно: не включай SELECT-запросы в скрипт — только CREATE TABLE и INSERT.\n"
     "Данные должны быть осмысленными и покрывать крайние случаи.\n"
     f"Крайние случаи которые стоит учитывать:\n{CASE_HINTS}"
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent: RunnableSerializable = agent_prompt | llm.bind_tools(tools, tool_choice="auto")

In [20]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

name2tool = {tool.name: tool.func for tool in tools}

class SQLAgentExecutor:
    def __init__(self, max_iterations: int = 10):
        self.max_iterations = max_iterations
        self.chat_history = []
        self.agent = agent

    def invoke(self, input: str) -> dict:
        count = 0
        agent_scratchpad = []
        final_answer = ""

        while count < self.max_iterations:
            tool_call = agent.invoke({
                "input": input,
                "chat_history": self.chat_history,
                "agent_scratchpad": agent_scratchpad
            })

            agent_scratchpad.append(tool_call)

            if tool_call.tool_calls:
                for tool_call_obj in tool_call.tool_calls:
                    tool_name = tool_call_obj["name"]
                    tool_args = tool_call_obj["args"]
                    tool_call_id = tool_call_obj["id"]

                    tool_out = name2tool[tool_name](**tool_args)

                    tool_exec = ToolMessage(
                        content=f"{tool_out}",
                        tool_call_id=tool_call_id
                    )
                    agent_scratchpad.append(tool_exec)
                    
                    preview = str(tool_out)[:200]
                    print(f"{count}: {tool_name} → {preview}")

                count += 1

                if any(tc["name"] == "final_answer" for tc in tool_call.tool_calls):
                    final_tool_call = next(tc for tc in tool_call.tool_calls if tc["name"] == "final_answer")
                    final_answer = final_tool_call["args"]["answer"]
                    break
            else:
                final_answer = tool_call.content
                break

        self.chat_history.extend([
            HumanMessage(content=input),
            AIMessage(content=final_answer)
        ])

        return {"output": final_answer}

sql_agent = SQLAgentExecutor()

In [22]:
task1 = """
Схема:
  employer(employer_id BIGINT, name VARCHAR)
  vacancy(vacancy_id BIGINT, active BOOLEAN, employer_id BIGINT)

Задача:
  Мы хотим отправить рассылку всем работодателям,
  у которых не более 5 активных вакансий.
  Нужно вывести имена таких работодателей.

Сохрани тестовые данные в файл task1.sql
"""

result1 = sql_agent.invoke(task1)
print(result1["output"])

0: validate_sql → SQL корректен
1: save_sql_script → Скрипт сохранён в 'task1.sql'
2: final_answer → {'answer': 'Файл `task1.sql` сохранён с тестовыми данными.\n\n**Краткое описение тестовых данных:**\n\n| employer_id | name | active_vacancies | включается? |\n|---|---|---|---|\n| 1 | TechCorp | 5 | 
Файл `task1.sql` сохранён с тестовыми данными.

**Краткое описение тестовых данных:**

| employer_id | name | active_vacancies | включается? |
|---|---|---|---|
| 1 | TechCorp | 5 | да (граничное значение) |
| 2 | StartUp Inc | 4 | да |
| 3 | Enterprise Ltd | 0 | да (только неактивные вакансии) |
| 4 | Global Solutions | 5 | да (граничное значение) |
| 5 | MegaCorp | 6 | нет (>5) |
| 6 | SmallBiz | 3 | да |
| 7 | MediumCo | 5 | да (граничное) |
| 8 | Tiny Inc | 1 | да |
| 9 | Another Co | 2 | да |
| 10 | NoVacancies Ltd | 0 | да (нет ни одной вакансии — LEFT JOIN) |

**Покрытые крайние случаи:**
- employer с ровно 5 активными вакансиями (TechCorp, Global Solutions, MediumCo) — граничное ус

In [23]:
!head -n 20 task1.sql

CREATE TABLE employer (
  employer_id BIGINT,
  name VARCHAR
);

CREATE TABLE vacancy (
  vacancy_id BIGINT,
  active BOOLEAN,
  employer_id BIGINT
);

INSERT INTO employer (employer_id, name) VALUES
(1, 'TechCorp'),
(2, 'StartUp Inc'),
(3, 'Enterprise Ltd'),
(4, 'Global Solutions'),
(5, 'MegaCorp'),
(6, 'SmallBiz'),
(7, 'MediumCo'),
(8, 'Tiny Inc'),
